# From Drift to Model Selection

In Notebook 1, we identified **model drift** — situations where your LLM produces degraded, inconsistent, or incorrect outputs due to query phrasing, terminology changes, or model updates. When drift becomes unacceptable, or when you need better quality, lower costs, or improved safety guarantees, the solution is often to **migrate to a different model**.

But how do you choose the right model? With dozens of options available on Amazon Bedrock — Claude, Nova, GPT-OSS, Qwen, and more — selecting the best model for your use case requires **systematic, data-driven evaluation**.

This notebook introduces **LLM-as-a-Jury** methodology: using multiple judge models to score candidate responses, then aggregating their judgments to make objective model selection decisions.

---

# Model Selection Evaluation

This notebook helps you compare multiple Bedrock models using **LLM-as-a-Jury** methodology to select the best model for your use case.

## Methodology

1. **Multi-Judge Evaluation**: 2-3 judge models score responses on standardized metrics
2. **Majority Voting**: Final PASS/FAIL based on consensus across judges
3. **Comprehensive Metrics**: Quality + Safety + Performance + Cost
4. **Data-Driven Selection**: Objective comparison across all dimensions

## Evaluation Framework

![LLM-as-a-Jury Evaluation Framework](data/llm-as-judge-background.png)

## Quality Metrics (1-5 scale)

| Metric | Description |
|--------|-------------|
| Correctness | Factual accuracy of the response |
| Completeness | Coverage of all aspects of the question |
| Relevance | Focus on the question asked |
| Format | Structure and readability |
| Coherence | Logical organization and clarity |
| Following Instructions | Adherence to specific instructions |

---

## Setup

In [ ]:
import boto3
import json
import time
import numpy as np
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, field
from datetime import datetime
import concurrent.futures

# Add src to path for utilities
import sys
sys.path.insert(0, '.')

from src.utils import (
    create_bedrock_client,
    JudgeScore,
    EvaluationResult,
    ModelComparison,
    SafetyEvaluation,
    QUALITY_METRICS,
    evaluate_with_jury,
    evaluate_medical_safety,
    compute_model_cost,
    invoke_model_for_evaluation,
    run_model_evaluation,
    compare_models
)

# Initialize Bedrock client
REGION = "us-east-1"
bedrock = create_bedrock_client(REGION)

print("Setup complete")
print(f"  Region: {REGION}")
print(f"  Quality metrics: {', '.join(QUALITY_METRICS)}")

---

## Configure Models

Define the candidate models to evaluate and the judge models for the jury.

In [ ]:
# Candidate models to evaluate
CANDIDATE_MODELS = {
    "qwen-coder": {
        "id": "qwen.qwen3-coder-30b-a3b-v1:0",
        "name": "Qwen 3 Coder",
        "cost_per_1k_input": 0.00035,
        "cost_per_1k_output": 0.0004
    },
    "nova-lite": {
        "id": "us.amazon.nova-2-lite-v1:0",
        "name": "Amazon Nova 2 Lite",
        "cost_per_1k_input": 0.00006,
        "cost_per_1k_output": 0.00024
    },
    "gpt-oss": {
        "id": "openai.gpt-oss-20b-1:0",
        "name": "OpenAI GPT OSS",
        "cost_per_1k_input": 0.0003,
        "cost_per_1k_output": 0.0006
    },
    "claude-haiku": {
        "id": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
        "name": "Claude Haiku 4.5",
        "cost_per_1k_input": 0.0008,
        "cost_per_1k_output": 0.004
    }
}

# Judge models for LLM-as-a-Jury evaluation
# Using diverse models from different providers for robust evaluation
# Note: Some models require specific regions
JUDGE_CONFIGS = [
    {"id": "us.anthropic.claude-3-5-haiku-20241022-v1:0", "region": "us-east-1", "name": "Claude 3.5 Haiku"},
    {"id": "us.amazon.nova-pro-v1:0", "region": "us-east-1", "name": "Nova Pro"},
    {"id": "moonshotai.kimi-k2.5", "region": "us-east-2", "name": "Kimi K2.5"},
]

# Create bedrock clients for each region needed
BEDROCK_CLIENTS = {}
for judge in JUDGE_CONFIGS:
    region = judge["region"]
    if region not in BEDROCK_CLIENTS:
        BEDROCK_CLIENTS[region] = create_bedrock_client(region)

# Extract just the model IDs for compatibility
JUDGE_MODELS = [j["id"] for j in JUDGE_CONFIGS]

print("Candidate Models:")
for key, model in CANDIDATE_MODELS.items():
    print(f"  {key}: {model['name']} ({model['id']})")

print(f"\nJudge Models ({len(JUDGE_CONFIGS)}):")
for judge in JUDGE_CONFIGS:
    print(f"  - {judge['name']} ({judge['id']}) [{judge['region']}]")

---

## Test Dataset

Medical/Healthcare prompts with golden answers and critical entities for safety evaluation.

In [ ]:
# Load test dataset from file
with open('data/medical_evaluation_dataset.json', 'r') as f:
    TEST_DATASET = json.load(f)

print(f"Test Dataset: {len(TEST_DATASET)} prompts")
print("\nCategories:")
categories = {}
for item in TEST_DATASET:
    cat = item['category']
    categories[cat] = categories.get(cat, 0) + 1
for cat, count in categories.items():
    print(f"  - {cat}: {count}")

print("\nSample prompt:")
print(f"  ID: {TEST_DATASET[0]['id']}")
print(f"  Prompt: {TEST_DATASET[0]['prompt'][:80]}...")

---

## Run Inference

Invoke each candidate model on all test prompts, collecting responses and performance metrics.

In [ ]:
# Store all model responses
model_responses = {}  # model_key -> {prompt_id -> response_data}

print("Running inference on all models...")
print("=" * 80)

for model_key, model_config in CANDIDATE_MODELS.items():
    print(f"\n{model_config['name']} ({model_config['id']})")
    print("-" * 60)
    
    model_responses[model_key] = {}
    
    for test_case in TEST_DATASET:
        prompt_id = test_case['id']
        prompt = test_case['prompt']
        
        try:
            response_text, input_tokens, output_tokens, latency_ms, cost = invoke_model_for_evaluation(
                prompt=prompt,
                model_id=model_config['id'],
                bedrock_client=bedrock,
                region=REGION
            )
            
            model_responses[model_key][prompt_id] = {
                'response': response_text,
                'input_tokens': input_tokens,
                'output_tokens': output_tokens,
                'latency_ms': latency_ms,
                'cost': cost,
                'success': not response_text.startswith('Error:')
            }
            
            status = "✓" if not response_text.startswith('Error:') else "✗"
            print(f"  {status} {prompt_id}: {latency_ms:.0f}ms, {output_tokens} tokens")
            
        except Exception as e:
            model_responses[model_key][prompt_id] = {
                'response': f'Error: {str(e)}',
                'input_tokens': 0,
                'output_tokens': 0,
                'latency_ms': 0,
                'cost': 0,
                'success': False
            }
            print(f"  ✗ {prompt_id}: Error - {str(e)[:50]}")

print("\n" + "=" * 80)
print("Inference complete")

In [ ]:
# Show sample response comparison
print("SAMPLE RESPONSE COMPARISON: MED-001 (Metformin Side Effects)")
print("=" * 80)

for model_key, model_config in CANDIDATE_MODELS.items():
    resp = model_responses[model_key].get('MED-001', {})
    print(f"\n{model_config['name']}")
    print("-" * 60)
    response_text = resp.get('response', 'N/A')
    # Show first 400 chars
    print(response_text[:400] + ('...' if len(response_text) > 400 else ''))
    print(f"\n[Latency: {resp.get('latency_ms', 0):.0f}ms | Tokens: {resp.get('output_tokens', 0)} | Cost: ${resp.get('cost', 0):.6f}]")

---

## Judge Evaluation

Run LLM-as-a-Jury evaluation and medical safety checks on all responses.

In [ ]:
# Custom jury evaluation to handle multi-region judges
from src.utils import evaluate_with_single_judge, METRIC_PASS_THRESHOLD
import concurrent.futures

def evaluate_with_multi_region_jury(
    user_prompt: str,
    model_response: str,
    judge_configs: list,
    bedrock_clients: dict,
    golden_answer: str = None
):
    """Evaluate with judges that may be in different regions."""
    judge_scores = []
    
    # Run judges in parallel
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(judge_configs)) as executor:
        futures = {}
        for judge in judge_configs:
            client = bedrock_clients[judge["region"]]
            future = executor.submit(
                evaluate_with_single_judge,
                user_prompt,
                model_response,
                judge["id"],
                golden_answer,
                client,
                judge["region"]
            )
            futures[future] = judge
        
        for future in concurrent.futures.as_completed(futures):
            judge_scores.append(future.result())
    
    # Aggregate scores
    aggregated_scores = {}
    for metric in QUALITY_METRICS:
        scores = [js.scores.get(metric, 0) for js in judge_scores if js.scores.get(metric)]
        aggregated_scores[metric] = np.mean(scores) if scores else 0.0
    
    # Majority judgment with score-based fallback
    pass_votes = sum(1 for js in judge_scores if js.judgment.upper() == "PASS")
    avg_score = np.mean(list(aggregated_scores.values())) if aggregated_scores else 0.0
    min_score = min(aggregated_scores.values()) if aggregated_scores else 0.0
    
    if pass_votes >= len(judge_scores) / 2:
        majority_judgment = "PASS"
    elif avg_score >= METRIC_PASS_THRESHOLD and min_score >= 2:
        majority_judgment = "PASS"
    else:
        majority_judgment = "FAIL"
    
    return judge_scores, majority_judgment, aggregated_scores

# Store evaluation results
evaluation_results = {}  # model_key -> [EvaluationResult]

print("Running jury evaluation with 3 judges (multi-region)...")
print("=" * 80)

for model_key, model_config in CANDIDATE_MODELS.items():
    print(f"\nEvaluating {model_config['name']}...")
    evaluation_results[model_key] = []
    
    for test_case in TEST_DATASET:
        prompt_id = test_case['id']
        resp_data = model_responses[model_key].get(prompt_id, {})
        
        if not resp_data.get('success', False):
            print(f"  ⚠ {prompt_id}: Skipped (inference failed)")
            continue
        
        response_text = resp_data['response']
        
        # Run jury evaluation with multi-region support
        try:
            judge_scores, majority_judgment, aggregated_scores = evaluate_with_multi_region_jury(
                user_prompt=test_case['prompt'],
                model_response=response_text,
                judge_configs=JUDGE_CONFIGS,
                bedrock_clients=BEDROCK_CLIENTS,
                golden_answer=test_case.get('golden_answer')
            )
            
            # Run safety evaluation
            safety_eval = evaluate_medical_safety(
                response_text=response_text,
                critical_entities=test_case.get('critical_entities', [])
            )
            
            eval_result = EvaluationResult(
                prompt_id=prompt_id,
                model_response=response_text,
                judge_scores=judge_scores,
                majority_judgment=majority_judgment,
                aggregated_scores=aggregated_scores,
                ttlb_ms=resp_data['latency_ms'],
                response_cost=resp_data['cost'],
                safety_score=safety_eval.critical_info_score
            )
            
            evaluation_results[model_key].append(eval_result)
            
            # Show individual judge votes
            votes = [f"{j.judgment[0]}" for j in judge_scores]  # P or F
            status = "✓" if majority_judgment == "PASS" else "✗"
            avg_score = np.mean(list(aggregated_scores.values()))
            print(f"  {status} {prompt_id}: {majority_judgment} (avg: {avg_score:.2f}, votes: {'/'.join(votes)}, safety: {safety_eval.critical_info_score:.0%})")
            
        except Exception as e:
            print(f"  ⚠ {prompt_id}: Evaluation error - {str(e)[:50]}")

print("\n" + "=" * 80)
print("Evaluation complete")

---

## Aggregate Results

Build model comparison with LLM evaluation rates and aggregated scores.

In [ ]:
# Convert to format expected by compare_models
eval_results_by_model_id = {
    CANDIDATE_MODELS[model_key]['id']: results
    for model_key, results in evaluation_results.items()
}

# Generate comparison
comparison = compare_models(eval_results_by_model_id)

# Build summary data for visualization
summary_data = []

for model_key, model_config in CANDIDATE_MODELS.items():
    model_id = model_config['id']
    results = evaluation_results.get(model_key, [])
    
    if not results:
        continue
    
    # Calculate metrics
    avg_latency = np.mean([r.ttlb_ms for r in results])
    total_cost = sum(r.response_cost for r in results)
    avg_safety = np.mean([r.safety_score for r in results])
    
    # Support both old (pass_rates) and new (llm_eval_rates) attribute names
    if hasattr(comparison, 'llm_eval_rates'):
        llm_eval_rate = comparison.llm_eval_rates.get(model_id, 0)
    else:
        llm_eval_rate = comparison.pass_rates.get(model_id, 0)
    
    quality_scores = comparison.quality_scores.get(model_id, {})
    llm_quality_score = np.mean(list(quality_scores.values())) if quality_scores else 0
    
    summary_data.append({
        'model_key': model_key,
        'model_name': model_config['name'],
        'model_id': model_id,
        'llm_eval_rate': llm_eval_rate,
        'llm_quality_score': llm_quality_score,
        'avg_latency': avg_latency,
        'total_cost': total_cost,
        'avg_safety': avg_safety,
        'quality_scores': quality_scores,
        'is_recommended': model_id == comparison.recommended_model
    })

# Print summary table
print("MODEL EVALUATION SUMMARY")
print("=" * 100)
print(f"{'Model':<25} {'LLM Eval':<12} {'LLM Quality':<12} {'Avg Latency':<15} {'Total Cost':<12} {'Safety':<10}")
print("-" * 100)

for data in summary_data:
    rec = " ★" if data['is_recommended'] else ""
    print(f"{data['model_name']:<25} {data['llm_eval_rate']:.0%}{'':<7} {data['llm_quality_score']:.2f}/5{'':<5} {data['avg_latency']:.0f}ms{'':<8} ${data['total_cost']:.5f}{'':<5} {data['avg_safety']:.0%}{rec}")

print("=" * 100)
print(f"\n★ Recommended Model: {comparison.recommended_model}")

---

## Visualizations

### Quality Score Heatmap

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Dark theme settings
DARK_BG = "#161e2d"
GRID_COLOR = "#2a3a5a"
TEXT_COLOR = "#e0e0e0"

# Prepare heatmap data
model_names = [d['model_name'] for d in summary_data]
metrics = QUALITY_METRICS

# Build score matrix
z_data = []
for data in summary_data:
    row = [data['quality_scores'].get(metric, 0) for metric in metrics]
    z_data.append(row)

# Custom colorscale: dark red (low) -> orange -> green (high)
# Better contrast and readability
custom_colorscale = [
    [0.0, '#8B0000'],   # Dark red (score 1)
    [0.25, '#DC143C'],  # Crimson (score 2)
    [0.5, '#FF8C00'],   # Dark orange (score 3)
    [0.75, '#32CD32'],  # Lime green (score 4)
    [1.0, '#006400']    # Dark green (score 5)
]

# Create heatmap
fig = go.Figure(data=go.Heatmap(
    z=z_data,
    x=[m.replace('_', ' ').title() for m in metrics],
    y=model_names,
    colorscale=custom_colorscale,
    zmin=1,
    zmax=5,
    text=[[f"{val:.2f}" for val in row] for row in z_data],
    texttemplate="%{text}",
    textfont={"size": 12, "color": "white"},
    hovertemplate="Model: %{y}<br>Metric: %{x}<br>Score: %{z:.2f}<extra></extra>",
    colorbar=dict(title="Score", tickvals=[1, 2, 3, 4, 5])
))

fig.update_layout(
    title=dict(
        text="Quality Scores by Model and Metric",
        font=dict(size=18, color=TEXT_COLOR)
    ),
    template="plotly_dark",
    paper_bgcolor=DARK_BG,
    plot_bgcolor=DARK_BG,
    height=400,
    margin=dict(l=150, r=50, t=80, b=80)
)

fig.show()

### Radar Chart: Multi-Dimensional Model Comparison

In [ ]:
# Radar chart for multi-dimensional comparison
fig = go.Figure()

# Radar dimensions (normalized to 0-1 scale)
radar_metrics = ['LLM Eval', 'LLM Quality', 'Safety', 'Speed', 'Cost Efficiency']

# Calculate speed and cost efficiency (inverted: lower is better -> higher score)
max_latency = max(d['avg_latency'] for d in summary_data) if summary_data else 1
max_cost = max(d['total_cost'] for d in summary_data) if summary_data else 1

colors = px.colors.qualitative.Set2

for i, data in enumerate(summary_data):
    values = [
        data['llm_eval_rate'],                       # LLM Eval Rate (0-1)
        data['llm_quality_score'] / 5,               # LLM Quality (normalized to 0-1)
        data['avg_safety'],                          # Safety (0-1)
        1 - (data['avg_latency'] / max_latency),     # Speed (inverted)
        1 - (data['total_cost'] / max_cost) if max_cost > 0 else 1  # Cost efficiency (inverted)
    ]
    # Close the polygon
    values.append(values[0])
    
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=radar_metrics + [radar_metrics[0]],
        fill='toself',
        name=data['model_name'],
        line=dict(color=colors[i % len(colors)]),
        opacity=0.7
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 1],
            tickvals=[0.2, 0.4, 0.6, 0.8, 1.0],
            gridcolor=GRID_COLOR
        ),
        angularaxis=dict(gridcolor=GRID_COLOR)
    ),
    title=dict(
        text="Multi-Dimensional Model Comparison",
        font=dict(size=18, color=TEXT_COLOR)
    ),
    template="plotly_dark",
    paper_bgcolor=DARK_BG,
    showlegend=True,
    legend=dict(x=1.1, y=0.5),
    height=500
)

fig.show()

### Cost vs Performance Scatter Plot

In [ ]:
# Scatter plot: Latency vs Cost, color by Safety
fig = go.Figure()

for i, data in enumerate(summary_data):
    # Size based on LLM eval rate
    size = 20 + data['llm_eval_rate'] * 40
    
    # Color based on safety score (green = safe, red = unsafe)
    safety_color = f"rgb({int(255 * (1 - data['avg_safety']))}, {int(255 * data['avg_safety'])}, 100)"
    
    fig.add_trace(go.Scatter(
        x=[data['avg_latency']],
        y=[data['total_cost'] * 1000],  # Convert to cost per 1K for readability
        mode='markers+text',
        marker=dict(
            size=size,
            color=data['avg_safety'],
            colorscale='RdYlGn',
            cmin=0,
            cmax=1,
            line=dict(width=2, color='white')
        ),
        text=[data['model_name']],
        textposition='top center',
        textfont=dict(color=TEXT_COLOR, size=11),
        name=data['model_name'],
        hovertemplate=(
            f"<b>{data['model_name']}</b><br>" +
            f"Latency: {data['avg_latency']:.0f}ms<br>" +
            f"Cost: ${data['total_cost']:.5f}<br>" +
            f"Safety: {data['avg_safety']:.0%}<br>" +
            f"LLM Eval: {data['llm_eval_rate']:.0%}" +
            "<extra></extra>"
        )
    ))

fig.update_layout(
    title=dict(
        text="Cost vs Latency (Bubble Size = LLM Eval Rate, Color = Safety)",
        font=dict(size=18, color=TEXT_COLOR)
    ),
    xaxis=dict(
        title="Average Latency (ms)",
        gridcolor=GRID_COLOR,
        color=TEXT_COLOR
    ),
    yaxis=dict(
        title="Cost per 1K Requests ($)",
        gridcolor=GRID_COLOR,
        color=TEXT_COLOR
    ),
    template="plotly_dark",
    paper_bgcolor=DARK_BG,
    plot_bgcolor=DARK_BG,
    height=500,
    showlegend=False
)

# Add annotation for ideal quadrant
fig.add_annotation(
    x=min(d['avg_latency'] for d in summary_data) * 0.8 if summary_data else 100,
    y=min(d['total_cost'] for d in summary_data) * 1000 * 0.8 if summary_data else 0.1,
    text="← Ideal (Fast + Cheap)",
    showarrow=False,
    font=dict(color="#90EE90", size=12)
)

fig.show()

### Summary Table

In [ ]:
# Create summary table with Plotly
header_values = ['Model', 'LLM Eval Rate', 'LLM Quality', 'Safety', 'Latency (ms)', 'Cost ($)', 'Recommended']

cell_values = [
    [d['model_name'] for d in summary_data],
    [f"{d['llm_eval_rate']:.0%}" for d in summary_data],
    [f"{d['llm_quality_score']:.2f}/5" for d in summary_data],
    [f"{d['avg_safety']:.0%}" for d in summary_data],
    [f"{d['avg_latency']:.0f}" for d in summary_data],
    [f"${d['total_cost']:.5f}" for d in summary_data],
    ["★ YES" if d['is_recommended'] else "" for d in summary_data]
]

# Highlight recommended model row
fill_colors = []
for i, d in enumerate(summary_data):
    if d['is_recommended']:
        fill_colors.append('#1a5f1a')  # Dark green for recommended
    else:
        fill_colors.append('#1e2a3a')

fig = go.Figure(data=[go.Table(
    header=dict(
        values=[f"<b>{h}</b>" for h in header_values],
        fill_color='#0d1520',
        align='center',
        font=dict(color=TEXT_COLOR, size=13),
        height=35
    ),
    cells=dict(
        values=cell_values,
        fill_color=[fill_colors for _ in header_values],
        align='center',
        font=dict(color=TEXT_COLOR, size=12),
        height=30
    )
)])

fig.update_layout(
    title=dict(
        text="Model Evaluation Summary",
        font=dict(size=18, color=TEXT_COLOR)
    ),
    paper_bgcolor=DARK_BG,
    height=250,
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.show()

---

## Final Recommendation

In [ ]:
# Generate final recommendation
print("=" * 80)
print("                    MODEL SELECTION RECOMMENDATION")
print("=" * 80)

# Find recommended model data
recommended = next((d for d in summary_data if d['is_recommended']), None)

if recommended:
    print(f"\n★ RECOMMENDED MODEL: {recommended['model_name']}")
    print(f"   Model ID: {recommended['model_id']}")
    print()
    print("   Performance Summary:")
    print(f"   ├─ LLM Eval Rate: {recommended['llm_eval_rate']:.0%}")
    print(f"   ├─ LLM Quality Score: {recommended['llm_quality_score']:.2f}/5")
    print(f"   ├─ Safety Score: {recommended['avg_safety']:.0%}")
    print(f"   ├─ Average Latency: {recommended['avg_latency']:.0f}ms")
    print(f"   └─ Total Cost (5 prompts): ${recommended['total_cost']:.5f}")
    
    print("\n   Quality Breakdown:")
    for metric, score in recommended['quality_scores'].items():
        bar = "█" * int(score) + "░" * (5 - int(score))
        print(f"   ├─ {metric.replace('_', ' ').title():<22} {bar} {score:.2f}")
    
    # Cost projection
    avg_cost_per_request = recommended['total_cost'] / len(TEST_DATASET)
    monthly_cost_100k = avg_cost_per_request * 100_000
    
    print(f"\n   Cost Projection:")
    print(f"   ├─ Per Request: ${avg_cost_per_request:.6f}")
    print(f"   └─ Monthly (100K requests): ${monthly_cost_100k:.2f}")
else:
    print("\n⚠ No model could be recommended. All models may have failed evaluation.")

print("\n" + "=" * 80)

# Comparison with alternatives
print("\nALTERNATIVES COMPARISON:")
print("-" * 80)

for data in sorted(summary_data, key=lambda x: (-x['llm_eval_rate'], -x['llm_quality_score'])):
    if data['is_recommended']:
        continue
    
    # Calculate tradeoffs vs recommended
    if recommended:
        eval_diff = data['llm_eval_rate'] - recommended['llm_eval_rate']
        quality_diff = data['llm_quality_score'] - recommended['llm_quality_score']
        cost_diff = data['total_cost'] - recommended['total_cost']
        latency_diff = data['avg_latency'] - recommended['avg_latency']
        
        print(f"\n{data['model_name']}:")
        print(f"  LLM Eval Rate: {data['llm_eval_rate']:.0%} ({eval_diff:+.0%} vs recommended)")
        print(f"  LLM Quality: {data['llm_quality_score']:.2f} ({quality_diff:+.2f})")
        print(f"  Latency: {data['avg_latency']:.0f}ms ({latency_diff:+.0f}ms)")
        print(f"  Cost: ${data['total_cost']:.5f} ({'+' if cost_diff > 0 else ''}{cost_diff:.5f})")

print("\n" + "=" * 80)

---

## Summary: Model Evaluation Checklist

### Before Evaluation

- [ ] Define candidate models to evaluate
- [ ] Select judge models (2-3 for robust consensus)
- [ ] Prepare test dataset with golden answers
- [ ] Identify critical entities for safety checks

### During Evaluation

- [ ] Run inference on all candidate models
- [ ] Execute LLM-as-a-Jury evaluation
- [ ] Check medical safety (critical entities, harmful advice)
- [ ] Collect performance metrics (latency, cost)

### After Evaluation

- [ ] Review quality score heatmap
- [ ] Analyze radar chart for trade-offs
- [ ] Consider cost vs performance scatter
- [ ] Validate recommended model selection
- [ ] Project monthly costs at scale

---

### Key Metrics

| Metric | Threshold | Notes |
|--------|-----------|-------|
| LLM Eval Rate | >= 80% | Majority of judges approve |
| LLM Quality Score | >= 3/5 | Average across all metrics |
| Safety Score | >= 80% | Critical entities present |
| Latency | Use case dependent | Consider user experience |
| Cost | Budget dependent | Project at expected scale |

---

**Congratulations!** You've completed the Model Selection Evaluation.

You now have data-driven insights to select the best model for your healthcare/medical use case based on:
- **Quality**: How well the model answers questions
- **Safety**: Critical information accuracy
- **Performance**: Latency and throughput
- **Cost**: Budget implications at scale

---

**Next**: Proceed to Notebook 3 (Prompt Optimization) to learn how to write prompts that work consistently across different models.